# 01 - Data Cleaning and Merging
The purpose of this notebook is to load, clean, and, merge the raw workout and bodyweight datasets. The primary steps include standardizing data types, handling missing values, and combining the two datasets on the "Date" column. The final output is a clean, master DataFrame that will be used for subsequent in-depth analysis and visualization.

In [176]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.max_rows', None)  # Display all rows
pd.set_option('display.max_colwidth', None)  # Increase column width
pd.set_option('display.float_format', '{:.2f}'.format)  # Format floats to 2 decimal places 

In [177]:
# loading csv files
workouts_df = pd.read_csv('../data/workout_data.csv')
weight_df = pd.read_csv('../data/measurement_data.csv')

### 2.1 - Previewing the Workout Data
To get a better idea of the way this data is organized, let's view the head along with any attributes with null values. This will help me decide what data I can safely delete.

In [178]:
# view first few rows
workouts_df.head()

,title,start_time,end_time,description,exercise_title,superset_id,exercise_notes,set_index,set_type,weight_lbs,reps,distance_miles,duration_seconds,rpe
0,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",it was hella busy so we gave up like a nafis workout,Single Leg Press (Machine),0.00,egym lowkey feel lighter,0,normal,50.00,8.00,NaN,NaN,8.50
1,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",it was hella busy so we gave up like a nafis workout,Single Leg Press (Machine),0.00,egym lowkey feel lighter,1,normal,80.00,4.00,NaN,NaN,10.00
2,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",it was hella busy so we gave up like a nafis workout,Leg Press Horizontal (Machine),0.00,egym,0,normal,200.00,8.00,NaN,NaN,8.00
3,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",it was hella busy so we gave up like a nafis workout,Leg Press Horizontal (Machine),0.00,egym,1,normal,250.00,8.00,NaN,NaN,9.00
4,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",it was hella busy so we gave up like a nafis workout,Seated Leg Curl (Machine),NaN,NaN,0,warmup,50.00,8.00,NaN,NaN,6.00


In [179]:
# checking data types
workouts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2104 entries, 0 to 2103
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             2104 non-null   object 
 1   start_time        2104 non-null   object 
 2   end_time          2104 non-null   object 
 3   description       314 non-null    object 
 4   exercise_title    2104 non-null   object 
 5   superset_id       78 non-null     float64
 6   exercise_notes    130 non-null    object 
 7   set_index         2104 non-null   int64  
 8   set_type          2104 non-null   object 
 9   weight_lbs        1957 non-null   float64
 10  reps              2042 non-null   float64
 11  distance_miles    11 non-null     float64
 12  duration_seconds  62 non-null     float64
 13  rpe               2004 non-null   float64
dtypes: float64(6), int64(1), object(7)
memory usage: 230.3+ KB


In [180]:
workouts_df.isnull().sum()

title                  0
start_time             0
end_time               0
description         1790
exercise_title         0
superset_id         2026
exercise_notes      1974
set_index              0
set_type               0
weight_lbs           147
reps                  62
distance_miles      2093
duration_seconds    2042
rpe                  100
dtype: int64

### 2.2 - Removing Metadata
We can immediately see some metadata that will not be necessary in our data analysis, so we can remove them. This includes the description and exercise notes. The title of the workout itself is also not strictly necessary in the data analysis, but may make it easier to organize specific exercises done on each day into their respective muscle groups.

In [181]:
workouts_df = workouts_df.drop(['description', 'exercise_notes'], axis=1)

### 2.3 - Verifying Irrelevant Exercise Data
Since we aim to look at strength training data, we can get rid of any data related to cardio or other exercise types. These exercises will not have weight or rep information. However, some exercises, like pull ups, may not have weight information since it is based on bodyweight. These exercises are vital for checking trends in strength so they should not be deleted. Thus, we will analyze the data closer to verify what data should and should not be deleted.

In [182]:
workouts_df[workouts_df['weight_lbs'].isnull()]

,title,start_time,end_time,exercise_title,superset_id,set_index,set_type,weight_lbs,reps,distance_miles,duration_seconds,rpe
7,Legs,"31 Aug 2025, 17:40","31 Aug 2025, 18:33",Spinning,NaN,0,normal,NaN,NaN,NaN,311.00,NaN
18,Arms,"28 Aug 2025, 12:42","28 Aug 2025, 13:30",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
19,Arms,"28 Aug 2025, 12:42","28 Aug 2025, 13:30",Chin Up,NaN,0,normal,NaN,12.00,NaN,NaN,8.50
20,Arms,"28 Aug 2025, 12:42","28 Aug 2025, 13:30",Chin Up,NaN,1,normal,NaN,8.00,NaN,NaN,9.50
41,Legs,"22 Aug 2025, 16:17","22 Aug 2025, 16:49",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
52,Chest/Back,"19 Aug 2025, 18:11","19 Aug 2025, 18:55",Pull Up,NaN,0,normal,NaN,10.00,NaN,NaN,9.00
53,Chest/Back,"19 Aug 2025, 18:11","19 Aug 2025, 18:55",Pull Up,NaN,1,normal,NaN,7.00,NaN,NaN,10.00
63,early bird gets the milkshake,"16 Aug 2025, 14:25","16 Aug 2025, 15:47",Chin Up,NaN,0,normal,NaN,12.00,NaN,NaN,9.50
64,early bird gets the milkshake,"16 Aug 2025, 14:25","16 Aug 2025, 15:47",Chin Up,NaN,1,normal,NaN,7.00,NaN,NaN,9.50
98,Arms,"7 Aug 2025, 15:04","7 Aug 2025, 16:36",Chin Up,NaN,0,normal,NaN,12.00,NaN,NaN,9.00


In [183]:
workouts_df[workouts_df['reps'].isnull()]

,title,start_time,end_time,exercise_title,superset_id,set_index,set_type,weight_lbs,reps,distance_miles,duration_seconds,rpe
7,Legs,"31 Aug 2025, 17:40","31 Aug 2025, 18:33",Spinning,NaN,0,normal,NaN,NaN,NaN,311.00,NaN
18,Arms,"28 Aug 2025, 12:42","28 Aug 2025, 13:30",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
41,Legs,"22 Aug 2025, 16:17","22 Aug 2025, 16:49",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
136,Arms,"2 Aug 2025, 15:38","2 Aug 2025, 17:08",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
165,Arms,"30 Jul 2025, 15:06","30 Jul 2025, 16:13",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
180,Chest/Back,"27 Jul 2025, 16:35","27 Jul 2025, 17:49",Spinning,NaN,0,normal,NaN,NaN,NaN,180.00,NaN
208,chesticles or back tickles?,"24 Jul 2025, 14:57","24 Jul 2025, 16:04",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
260,Arms,"14 Jul 2025, 15:29","14 Jul 2025, 16:31",Spinning,NaN,0,normal,NaN,NaN,NaN,510.00,NaN
272,Chest/Back,"8 Jul 2025, 15:39","8 Jul 2025, 16:15",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
295,little bit of this little bit of that nasar chowdhury,"25 Jun 2025, 16:19","25 Jun 2025, 16:59",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN


### 2.4 - Deleting Irrelevant Exercise Data
Since I have verified that bodyweight exercises do not have weight information, I will remove exercises based on other attributes, like distance and duration. Then, we can check back and make sure that there are no remaining irrelevant exercises in the dataset and look at the full datasets data types using info().

In [184]:
workouts_df[workouts_df['distance_miles'].notna()]

,title,start_time,end_time,exercise_title,superset_id,set_index,set_type,weight_lbs,reps,distance_miles,duration_seconds,rpe
1486,Core/Cardio,"27 Nov 2024, 08:03","27 Nov 2024, 09:18",Treadmill,NaN,0,normal,NaN,NaN,0.52,1074.00,NaN
1514,Afternoon workout 💪,"25 Nov 2024, 16:44","25 Nov 2024, 17:19",Treadmill,NaN,0,normal,NaN,NaN,0.75,1250.00,NaN
1594,Core/Cardio,"21 Nov 2024, 07:31","21 Nov 2024, 08:45",Treadmill,NaN,0,normal,NaN,NaN,0.97,1505.00,NaN
1638,Evening workout 🏋️,"18 Nov 2024, 17:57","18 Nov 2024, 19:01",Treadmill,NaN,0,normal,NaN,NaN,1.74,2710.00,NaN
1675,Afternoon workout 💪,"16 Nov 2024, 16:44","16 Nov 2024, 18:05",Treadmill,NaN,0,normal,NaN,NaN,1.51,2400.00,NaN
1692,Evening workout 🏋️,"15 Nov 2024, 19:26","15 Nov 2024, 20:22",Treadmill,NaN,0,normal,NaN,NaN,1.15,1815.00,NaN
1737,Chest/Back,"14 Nov 2024, 10:07","14 Nov 2024, 11:46",Treadmill,NaN,0,normal,NaN,NaN,0.20,365.00,NaN
1753,Core/Cardio,"13 Nov 2024, 10:51","13 Nov 2024, 12:02",Treadmill,NaN,0,normal,NaN,NaN,0.82,1280.00,NaN
1900,Core/Cardio,"30 Oct 2024, 12:56","30 Oct 2024, 14:01",Treadmill,NaN,0,normal,NaN,NaN,0.25,572.00,NaN
2040,Core/Cardio,"7 Oct 2024, 19:58","7 Oct 2024, 20:54",Treadmill,NaN,0,normal,NaN,NaN,0.32,900.00,NaN


In [185]:
workouts_df[workouts_df['duration_seconds'].notna()]

,title,start_time,end_time,exercise_title,superset_id,set_index,set_type,weight_lbs,reps,distance_miles,duration_seconds,rpe
7,Legs,"31 Aug 2025, 17:40","31 Aug 2025, 18:33",Spinning,NaN,0,normal,NaN,NaN,NaN,311.00,NaN
18,Arms,"28 Aug 2025, 12:42","28 Aug 2025, 13:30",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
41,Legs,"22 Aug 2025, 16:17","22 Aug 2025, 16:49",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
136,Arms,"2 Aug 2025, 15:38","2 Aug 2025, 17:08",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
165,Arms,"30 Jul 2025, 15:06","30 Jul 2025, 16:13",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
180,Chest/Back,"27 Jul 2025, 16:35","27 Jul 2025, 17:49",Spinning,NaN,0,normal,NaN,NaN,NaN,180.00,NaN
208,chesticles or back tickles?,"24 Jul 2025, 14:57","24 Jul 2025, 16:04",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
260,Arms,"14 Jul 2025, 15:29","14 Jul 2025, 16:31",Spinning,NaN,0,normal,NaN,NaN,NaN,510.00,NaN
272,Chest/Back,"8 Jul 2025, 15:39","8 Jul 2025, 16:15",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
295,little bit of this little bit of that nasar chowdhury,"25 Jun 2025, 16:19","25 Jun 2025, 16:59",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN


In [186]:
workouts_df = workouts_df[workouts_df['distance_miles'].isnull()]
workouts_df = workouts_df[workouts_df['duration_seconds'].isnull()]

In [187]:
workouts_df[workouts_df['reps'].isnull()]

,title,start_time,end_time,exercise_title,superset_id,set_index,set_type,weight_lbs,reps,distance_miles,duration_seconds,rpe


In [188]:
workouts_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2042 entries, 0 to 2103
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             2042 non-null   object 
 1   start_time        2042 non-null   object 
 2   end_time          2042 non-null   object 
 3   exercise_title    2042 non-null   object 
 4   superset_id       78 non-null     float64
 5   set_index         2042 non-null   int64  
 6   set_type          2042 non-null   object 
 7   weight_lbs        1957 non-null   float64
 8   reps              2042 non-null   float64
 9   distance_miles    0 non-null      float64
 10  duration_seconds  0 non-null      float64
 11  rpe               2004 non-null   float64
dtypes: float64(6), int64(1), object(5)
memory usage: 207.4+ KB


### 2.5 - Final Removal of Irrelevant Columns
Since all our cardio workouts have been removed from the dataset, we can also remove the columns that track distance and duration. Additionally, since it is not particularly pertinent to this analysis, we can also get rid of superset information and RPE.

In [189]:
workouts_df = workouts_df.drop(['superset_id', 'distance_miles', 'duration_seconds', 'rpe'], axis=1)

In [190]:
workouts_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2042 entries, 0 to 2103
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           2042 non-null   object 
 1   start_time      2042 non-null   object 
 2   end_time        2042 non-null   object 
 3   exercise_title  2042 non-null   object 
 4   set_index       2042 non-null   int64  
 5   set_type        2042 non-null   object 
 6   weight_lbs      1957 non-null   float64
 7   reps            2042 non-null   float64
dtypes: float64(2), int64(1), object(5)
memory usage: 143.6+ KB
